In [114]:
!pip install svgpathtools

In [7]:
import pandas as pd
import json
import math
from fontTools.ttLib import TTFont
from fontTools.pens.basePen import BasePen

In [15]:
INPUT_FILE = r"C:\Users\mikha\Customers\Autura\lots.xlsx" 
SHEET_NAME = "Sheet0"             # If XLSX has multiple sheets
OUTPUT_GEOJSON_SHAPES = r"C:\Users\mikha\Customers\Autura\parking_shapes.geojson"
OUTPUT_GEOJSON_LABELS = r"C:\Users\mikha\Customers\Autura\parking_labels.geojson"


In [9]:
STALL_WIDTH = 40
STALL_HEIGHT = 80
GAP = 10
COLUMNS = 30
SECTION_GAP_Y = 100
LOT_GAP_X = 3000
TOP_PADDING = 100
LEFT_PADDING = 100
FONT_PATH = r"C:\Windows\Fonts\ARIALN.TTF"  # Path to a TTF font on your system


In [10]:
class GlyphToCoordsPen(BasePen):
    def __init__(self, glyphSet):
        BasePen.__init__(self, glyphSet)
        self.paths = []
        self.currentPath = []

    def _moveTo(self, p0):
        if self.currentPath:
            self.paths.append(self.currentPath)
        self.currentPath = [list(p0)]

    def _lineTo(self, p1):
        self.currentPath.append(list(p1))

    def _qCurveToOne(self, p1, p2):
        self.currentPath.append(list(p2))

    def _curveToOne(self, p1, p2, p3):
        self.currentPath.append(list(p3))

    def _closePath(self):
        if self.currentPath:
            self.paths.append(self.currentPath)
            self.currentPath = []

    def getPaths(self):
        if self.currentPath:
            self.paths.append(self.currentPath)
            self.currentPath = []
        return self.paths

In [11]:
df = pd.read_excel(INPUT_FILE, sheet_name=SHEET_NAME, usecols=[0,1,2,3])
df.columns = ["LotCode","Lotid","LotSection","LotStall"]
df["LotStall"] = df["LotStall"].astype(str).str.replace("Space", "").str.strip()
df = df.drop_duplicates(subset=["LotCode","LotSection","LotStall"]).reset_index(drop=True)
df = df.sort_values(["LotCode","LotSection","LotStall"]).reset_index(drop=True)

In [12]:
font = TTFont(FONT_PATH)
glyph_set = font.getGlyphSet()
cmap = font.getBestCmap()
em = font['head'].unitsPerEm

In [13]:
shape_features = []
label_features = []

lot_codes = df["LotCode"].unique()

for lot_idx, lot_code in enumerate(lot_codes):
    lot_df = df[df["LotCode"] == lot_code]
    lot_x = LEFT_PADDING + lot_idx * LOT_GAP_X
    current_y = TOP_PADDING

    for section in lot_df["LotSection"].unique():
        section_df = lot_df[lot_df["LotSection"] == section].reset_index(drop=True)
        num_stalls = len(section_df)
        rows_needed = math.ceil(num_stalls / COLUMNS)

        for i, row in section_df.iterrows():
            col = i % COLUMNS
            r = i // COLUMNS
            x = lot_x + col * (STALL_WIDTH + GAP)
            y = current_y + r * (STALL_HEIGHT + GAP)

            stall_id = row.LotStall
            geo_id = f"{row.LotCode}_{row.LotSection}_{stall_id}"

            # ------------------------
            # Rectangle feature (_GEOID_)
            # ------------------------
            poly_coords = [
                [x, y],
                [x + STALL_WIDTH, y],
                [x + STALL_WIDTH, y + STALL_HEIGHT],
                [x, y + STALL_HEIGHT],
                [x, y]
            ]
            shape_features.append({
                "type": "Feature",
                "geometry": {"type": "Polygon", "coordinates": [poly_coords]},
                "properties": {"GEOID": geo_id}
            })

            # ------------------------
            # Label polygon feature
            # ------------------------
            cx = x + STALL_WIDTH / 2
            cy = y + STALL_HEIGHT / 2

            # Prepare glyphs for the label
            glyphs = []
            for char in stall_id:
                glyph_index = cmap.get(ord(char))
                if glyph_index is None:
                    continue
                g = glyph_set[glyph_index]
                if g.width > 0:
                    glyphs.append(g)
            if not glyphs:
                continue  # skip empty label

            total_width_units = sum([g.width for g in glyphs])
            if total_width_units == 0:
                total_width_units = 1  # avoid division by zero

            # Scale to fit inside rectangle
            scale_x = STALL_WIDTH * 0.8 / total_width_units
            scale_y = STALL_HEIGHT * 0.8 / em
            scale = min(scale_x, scale_y)

            # Starting cursor (top-left of first glyph)
            cursor_x = cx - total_width_units * scale / 2
            cursor_y = cy - em * scale / 2

            label_paths = []
            for glyph in glyphs:
                pen = GlyphToCoordsPen(glyph_set)
                glyph.draw(pen)
                for path in pen.getPaths():
                    scaled_path = [[cursor_x + px*scale, cursor_y + py*scale] for px, py in path]
                    label_paths.append(scaled_path)
                cursor_x += glyph.width * scale  # advance to next glyph

            # Add each glyph path as a polygon feature
            for path in label_paths:
                label_features.append({
                    "type": "Feature",
                    "geometry": {"type": "Polygon", "coordinates": [path]},
                    "properties": {"label": stall_id}
                })

        # Move down for next section
        current_y += rows_needed * (STALL_HEIGHT + GAP) + SECTION_GAP_Y

In [16]:
geojson_shapes = {"type": "FeatureCollection", "features": shape_features}
geojson_labels = {"type": "FeatureCollection", "features": label_features}

with open(OUTPUT_GEOJSON_SHAPES, "w") as f:
    json.dump(geojson_shapes, f, indent=2)

with open(OUTPUT_GEOJSON_LABELS, "w") as f:
    json.dump(geojson_labels, f, indent=2)
